In [ ]:
#Importar el Dataframe

from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)

print(housing.DESCR)

df = housing.frame
df

**Parte 1**

a) Limpieza de datos.

b) Análisis exploratorio de datos (EDA).

**Parte 2**

Correr un modelo de regresión lineal para predecir el valor de *MedHouseVal*.

**Parte 3**

Extraer conclusiones de la predicción y presentar los resultados.

# Parte 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Mapa de valores nulos")
plt.show()

### no hay valores nulos


In [ ]:
print(f"Tenemos {df.duplicated().sum()} filas duplicadas")

In [ ]:
df.columns

In [ ]:

df = df.drop('Longitude', axis=1)
df = df.drop('Latitude', axis=1)
df


In [ ]:
plt.hist(df['HouseAge'].dropna(), bins=50, edgecolor='k')
plt.title("Distribución de antiguedades")
plt.xlabel("Antiguedad")
plt.ylabel("Frecuencia")
valor_marcar = df['HouseAge'].mean()
plt.xticks(rotation = 0)
plt.axvline(x=valor_marcar, color='red', linestyle='--', linewidth=2, label=f"edad Promedio = {valor_marcar:.0f}     ")
valor_marcar2 = df['HouseAge'].mode()
print(valor_marcar2)
plt.xticks(rotation = 0)
plt.axvline(x=valor_marcar2[0], color='green', linestyle='--', linewidth=2, label=f"edad mas comun = {valor_marcar2[0]:.0f}     ")
plt.legend()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,
            cbar_kws={"label": "Correlación"},
            annot_kws={"size": 10, "color": "black"})
plt.title("Matriz de correlación entre variables numéricas", pad=20)
plt.tight_layout()
plt.show()


In [ ]:
cols = df.columns
for col in cols:
    plt.figure(figsize=(3,1), )
    sns.boxplot(x=df[col], showfliers= False)
    plt.title(f"Boxplot de {col} (sin outliers)")
    plt.show()

In [ ]:
valorRaro = df['MedHouseVal'].mode()
df_filtrado = df[df['MedHouseVal']!= valorRaro[0]]
df_filtrado = df_filtrado[df_filtrado['AveRooms']< 15]
muestra = df_filtrado.sample(n=500)
print(valorRaro)
muestra

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(x="MedInc", y="MedHouseVal", data=muestra , alpha=1)
sns.regplot(x="MedInc", y="MedHouseVal", data=muestra , scatter=False, color="red")
plt.title("Scatterplot: MedInc vs MedHouseVal")
plt.xlabel("MedInc x$100k")
plt.ylabel("MedHouseVal (Valor medio de la casa) x$100k")
plt.show()

In [ ]:
sns.scatterplot(x="AveRooms", y="AveBedrms", data=muestra, alpha=1)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
y = df[["MedHouseVal"]]
X = df.drop(columns=["MedHouseVal"])
X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2)
reg = LinearRegression()
reg.fit(X_train,Y_train)
estimated = [reg.intercept_, reg.coef_]
print(estimated)

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Predicción y línea teórica
ypred = reg.predict(X_test)  # Solo si es necesario
plt.scatter(Y_test, ypred, label='Teórica', color='blue')
min_val = min(np.min(Y_test), np.min(ypred))
max_val = max(np.max(Y_test), np.max(ypred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Línea 45°')

# Etiquetas y leyenda
plt.xlabel('Valor real', fontsize=12)
plt.ylabel('Valor predicho', fontsize=12)
plt.title('Valor real vs valor predicho')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()



In [ ]:

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

# Paso 1: Reducir X_train a 1 dimensión con PCA
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_train)

# Paso 3: Entrenar regresión lineal sobre componente principal
reg = LinearRegression()
reg.fit(X_pca, Y_train)

# Predicción
y_pred = reg.predict(X_pca)

# Ordenar para graficar correctamente
sorted_idx = np.argsort(X_pca.ravel())
X_sorted = X_pca.ravel()[sorted_idx]
y_sorted = y_pred[sorted_idx]

# Paso 4: Graficar
plt.figure(figsize=(10, 6))
plt.scatter(X_pca, Y_train, label='Datos reales (PCA)', color='gray')
plt.plot(X_sorted, y_sorted, label='Regresión (PCA)', color='blue')

plt.xlabel('Primera Componente Principal (PCA)', fontsize=15)
plt.ylabel('Variable objetivo', fontsize=15)
plt.legend()
plt.title('Regresión lineal sobre primera componente PCA')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

# Predicciones en el test
X_test_pca = pca.transform(X_test)
y_test_pred = reg.predict(X_test_pca)

# Calcular ECM
mse = mean_squared_error(Y_test, y_test_pred)
print("Error Cuadrático Medio (MSE):", mse)
print(f'En promedio el test erra por U$D {np.sqrt(mse)*100000:,.2f}')

R2 = reg.score(ypred,Y_test)
print(f'ruido = {R2}')

**Se puede ver como se estima relativamente bien el valor promedio de las casas con un ruido bajo**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
y = df[["Population"]]
X = df.drop(columns=["Population"])
X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2)
reg = LinearRegression()
reg.fit(X_train,Y_train)
estimated = [reg.intercept_, reg.coef_]
print(estimated)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Predicción y línea teórica
ypred = reg.predict(X_test)  # Solo si es necesario
plt.scatter(Y_test, ypred, label='Teórica', color='blue')
min_val = min(np.min(Y_test), np.min(ypred))
max_val = max(np.max(Y_test), np.max(ypred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Línea 45°')

# Etiquetas y leyenda
plt.xlabel('Valor real', fontsize=12)
plt.ylabel('Valor predicho', fontsize=12)
plt.title('Valor real vs valor predicho')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

# Paso 1: Reducir X_train a 1 dimensión con PCA
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_train)

# Paso 3: Entrenar regresión lineal sobre componente principal
reg = LinearRegression()
reg.fit(X_pca, Y_train)

# Predicción
y_pred = reg.predict(X_pca)

# Ordenar para graficar correctamente
sorted_idx = np.argsort(X_pca.ravel())
X_sorted = X_pca.ravel()[sorted_idx]
y_sorted = y_pred[sorted_idx]

# Paso 4: Graficar
plt.figure(figsize=(10, 6))
plt.scatter(X_pca, Y_train, label='Datos reales (PCA)', color='gray')
plt.plot(X_sorted, y_sorted, label='Regresión (PCA)', color='blue')

plt.xlabel('Primera Componente Principal (PCA)', fontsize=15)
plt.ylabel('Variable objetivo', fontsize=15)
plt.legend()
plt.title('Regresión lineal sobre primera componente PCA')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

# Predicciones en el test
X_test_pca = pca.transform(X_test)
y_test_pred = reg.predict(X_test_pca)

# Calcular ECM
mse = mean_squared_error(Y_test, y_test_pred)
print("Error Cuadrático Medio (MSE):", mse)
print(f'En promedio el test erra por  {np.sqrt(mse):,.0f} habitantes')

R2 = reg.score(ypred,Y_test)
print(f'ruido = {R2}')

**En cambio los otros valores son malos para estimar la poblacion del bloque, logico sabiendo que son datos que no definen poblaciones**

***Hago una regresion simple para comparar el valor medio de un hogar unicamente con la edad de la casa***

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
y = df[["MedHouseVal"]]
X = df[["HouseAge"]]
X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2)
reg = LinearRegression()
reg.fit(X_train,Y_train)
estimated = [reg.intercept_, reg.coef_]
print(estimated)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Predicción y línea teórica
ypred = reg.predict(X_test)  # Solo si es necesario
plt.scatter(Y_test, ypred, label='Teórica', color='blue')
min_val = min(np.min(Y_test), np.min(ypred))
max_val = max(np.max(Y_test), np.max(ypred))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Línea 45°')

# Etiquetas y leyenda
plt.xlabel('Valor real', fontsize=12)
plt.ylabel('Valor predicho', fontsize=12)
plt.title('Valor real vs valor predicho')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

# Paso 1: Reducir X_train a 1 dimensión con PCA
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_train)

# Paso 3: Entrenar regresión lineal sobre componente principal
reg = LinearRegression()
reg.fit(X_pca, Y_train)

# Predicción
y_pred = reg.predict(X_pca)

# Ordenar para graficar correctamente
sorted_idx = np.argsort(X_pca.ravel())
X_sorted = X_pca.ravel()[sorted_idx]
y_sorted = y_pred[sorted_idx]

# Paso 4: Graficar
plt.figure(figsize=(10, 6))
plt.scatter(X_pca, Y_train, label='Datos reales (PCA)', color='gray')
plt.plot(X_sorted, y_sorted, label='Regresión (PCA)', color='blue')

plt.xlabel('Primera Componente Principal (PCA)', fontsize=15)
plt.ylabel('Variable objetivo', fontsize=15)
plt.legend()
plt.title('Regresión lineal sobre primera componente PCA')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

# Predicciones en el test
X_test_pca = pca.transform(X_test)
y_test_pred = reg.predict(X_test_pca)

# Calcular ECM
mse = mean_squared_error(Y_test, y_test_pred)
print("Error Cuadrático Medio (MSE):", mse)
print(f'En promedio el test erra por U$D {np.sqrt(mse)*100000:,.2f}')

R2 = reg.score(ypred,Y_test)
print(f'ruido = {R2}')

**No es mala forma de estimar**

In [ ]:
y_muestra = muestra[["MedHouseVal"]]
X_muestra = muestra.drop(columns=["MedHouseVal"])
X_muestra_train,X_muestra_test,Y_muestra_train,Y_muestra_test = train_test_split(X_muestra,y_muestra,test_size=0.2)
reg = LinearRegression()
reg.fit(X_muestra_train,Y_muestra_train)
estimated = [reg.intercept_, reg.coef_]
print(estimated)

In [ ]:
# Predicción y línea teórica
ypred_muestra = reg.predict(X_muestra_test)  # Solo si es necesario
plt.scatter(Y_muestra_test, ypred_muestra, label='Teórica', color='blue')
min_val = min(np.min(Y_muestra_test), np.min(ypred_muestra))
max_val = max(np.max(Y_muestra_test), np.max(ypred_muestra))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Línea 45°')

plt.xlabel('Valor real', fontsize=12)
plt.ylabel('Valor predicho', fontsize=12)
plt.title('Valor real vs valor predicho')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

# Paso 1: Reducir X_train a 1 dimensión con PCA
pca = PCA(n_components=1)
X_muestra_pca = pca.fit_transform(X_muestra_train)

# Paso 3: Entrenar regresión lineal sobre componente principal
reg = LinearRegression()
reg.fit(X_muestra_pca, Y_muestra_train)

# Predicción
y_pred = reg.predict(X_muestra_pca)

# Ordenar para graficar correctamente
sorted_idx = np.argsort(X_muestra_pca.ravel())
X_sorted = X_muestra_pca.ravel()[sorted_idx]
y_sorted = y_pred[sorted_idx]

# Paso 4: Graficar
plt.figure(figsize=(10, 6))
plt.scatter(X_muestra_pca, Y_muestra_train, label='Datos reales (PCA)', color='gray')
plt.plot(X_sorted, y_sorted, label='Regresión (PCA)', color='blue')

plt.xlabel('Primera Componente Principal (PCA)', fontsize=15)
plt.ylabel('Variable objetivo', fontsize=15)
plt.legend()
plt.title('Regresión lineal sobre primera componente PCA')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

# Predicciones en el test
X_test_pca = pca.transform(X_muestra_test)
y_test_pred = reg.predict(X_test_pca)

# Calcular ECM
mse = mean_squared_error(Y_muestra_test, y_test_pred)
print("Error Cuadrático Medio (MSE):", mse)
print(f'En promedio el test erra por U$D {np.sqrt(mse)*100000:,.2f}')

R2 = reg.score(ypred,Y_test)
print(f'ruido = {R2}')

**Se ve como el test aplicado a esta muestra es mas efectivo que el aplicado a la poblacion (no siempre va a ser asi)**